# 06 · Analysis (heavy computations)

**What this notebook does.** It computes, once per session, the data that the results notebook (07) reads: band power,
beta and gamma bursts, burst properties and averages, theta signals for phase coupling, and beta power. Each step loops over the 60 final
sessions and writes one file per session into `DATA_OUT`. These runs take hours (some overnight), so **they are only run when
`RUN = True`**; with `RUN = False` every cell just reports how many sessions are done.

**Run order in the project:** 01 → 02 → 03 → 04 → 05 → **06** → 07. Needs the tables of 01–05 (`final_sessions.csv`, `trials_filtered.csv`,
`extra_bad_channels.csv`, `channel_regions_all.csv`, `representative_channels.csv`).

**Steps** (colour of the heading = frequency band: green theta, blue beta / bursts, purple gamma, amber delta; no colour = all bands)

| Step | Computes | Output folder in `DATA_OUT` | Used in 07 |
|---|---|---|---|
| 1 | Band power, all bands, per trial and time course (CMR and LOCAL) | `band power\CMR`, `band power\LOCAL` | section 1 |
| 2 | Beta and gamma bursts, 3 representative channels, 50-ms bins | `Bursts detection` | section 3 |
| 3 | Beta bursts with exact start / end times | `Bursts detection exact times` | section 3 |
| 4 | Beta and gamma bursts on a grid over the whole probe | `Beta and gamma bursts across channels` | section 3 (other channels) |
| 5 | Figure: what the CMR and LOCAL references subtract | figure | methods |
| 6 | Burst properties at the 98th percentile (Karvat 2020) | `Burst properties p98` | section 3 (steps 8–10) |
| 7 | Average IL beta burst per session | `Burst averages p98 LOCAL` | section 3 (steps 9–10) |
| 8 | Theta signal of all channels (for any channel pair) | `Theta signal all channels\<ref>` | section 2 (all pairs) |
| 9 | Representative IL / PL / Cg1 signals in CMR and SR30-500 | `Theta coupling representative channels` | section 2 |
| 10 | Beta power normalised to the 90th percentile | `Beta power p90 LOCAL` | section 3 (beta power) |
| 11 | Channel selection for the all-pairs IL–PL test | table + figure | section 2 |
| 12 | Per-trial IL / PL theta phases (3 chosen pairs, SR30-500) | `Theta\Phase connectivity\Tests\Trial_phases_press_SR30-500_ERPkept` | section 2 (+ coupling along the session) |
| 13 | Jäckel Fig. 3C sums, ERP removed (δ, θ; press, release; correct, early) | `Tests\T8c_Jaeckel_Fig3C_replication_SR30-500_ERPremoved` | section 2 (T8c) |
| 14 | Overnight coupling tests T8 θ / δ, T11, T13–15 | `Tests\T8_*`, `T11_*`, `Overnight_T13-15_*` | section 2 (summaries, T14 cells) |
| 15 | T13 fix: coupling before the tone, correct vs late | `Tests\Overnight_T13-15_SR30-500_ERPkept\T13_rows_fix.csv` | section 2 (summaries) |
| 16 | Multi-unit firing over time (spike-rate curves) | `tables\mua_curves` | section 1 (high gamma + spikes) |

**Removed from the old notebook** (not used by any result in 07): three test cells on one session and the exact-time test; burst properties at
the 90th percentile (old version and p90 of the new one; set `PCTS = (98, 90)` to make p90 again); the 15–30 Hz burst averages (5f);
theta coupling of 3 pairs (5h); fast gamma coupling (5j, Watanabe 2021); bursts and power on two other IL / PL channels (5m).
Steps 12–16 were recovered from the old coupling / MUA cells; their code is in `fx_res_coupling.py` and `fx_res_power.py` (the same files as 07)
and they are run with `fxr.run(...)`, like the cells of 07.

**How to use it:** run the settings cell first; then any step. Change things only in the settings cell. Functions: `fx_analysis.py`
(+ `functions.py`); colours and saving: `fx_common.py`.

**The references:** **CMR** = channel − median of all channels; **LOCAL** = channel − median of the channels 30–200 µm away;
**SR30-500** = channel − median of the channels of the **same region** 30–500 µm away (used for coupling, step 9). Bad channels are removed first.

## Settings

**Paths:** `CODE_DIR` (notebooks + `fx_*.py` + `functions.py`), `DATA_ROOT`, `PROC` (sessions), `AMIR`, `TABLES` (temporarily the old folder),
`DATA_OUT` (all computed data, one sub-folder per step), `FIG_DIR` (figures), `PROBE_DIR` (the referencing figure, step 5).

**Switches:**
- `RUN = False`: every step only reports what is done. `True`: computes the missing sessions (hours; best overnight).
- `OVERWRITE = False`: finished sessions are skipped. `True`: recompute all.
- `SAVE = False`: figures and tables are not written (the data files of the steps are controlled by `RUN`). To save one cell's figure
  elsewhere, add `folder=r"...", force=True` to its save line.

**Data loaded here:** `final`, `XB` (extra bad channels), `TR` / `K` (all / kept trials), `REP` (fixed position of the representative channel per
rat and region, rule 1.13b), `CHREG` (region of every channel).

**Step parameters:** `TEST` (session for examples); `STEP_UM`, `MAX_HOURS` (step 4); `PCTS` (step 6); `REF_SES` (step 5).

In [ ]:
# ══ SETTINGS — run first ══
import os, sys, importlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore", message="The following channels did not have enough neighbors")
warnings.filterwarnings("ignore", message="Mean of empty slice|All-NaN slice")

# ── paths ──
CODE_DIR  = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"                  # folder with the notebooks + fx_*.py + functions.py
DATA_ROOT = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"             # project data on the server
PROC      = os.path.join(DATA_ROOT, "2_processed")                                 # 1000 Hz recordings
AMIR      = os.path.join(DATA_ROOT, "amir_data")                                   # your own data folder
TABLES    = os.path.join(CODE_DIR, "tables")                                       # TEMPORARY: old place; later os.path.join(AMIR, "tables")
DATA_OUT  = os.path.join(AMIR, "data")                                             # all computed data (one sub-folder per step)
FIG_DIR   = os.path.join(AMIR, "figures", "06_analysis")                           # figures of this notebook (if SAVE)
PROBE_DIR = os.path.join(AMIR, "Probe")                                            # referencing figure (step 5, if SAVE)

# ── output folders of the steps (change a name only if you also change it in 07) ──
BP_DIRS   = {"CMR": os.path.join(DATA_OUT, "band power", "CMR"), "LOCAL": os.path.join(DATA_OUT, "band power", "LOCAL")}   # step 1
BURST_DIR = os.path.join(DATA_OUT, "Bursts detection")                             # step 2
EXACT_DIR = os.path.join(DATA_OUT, "Bursts detection exact times")                 # step 3
ALL_DIR   = os.path.join(DATA_OUT, "Beta and gamma bursts across channels")        # step 4
PROP_ROOT = DATA_OUT                                                               # step 6 writes "Burst properties p<pct>" here
AVG_DIR   = os.path.join(DATA_OUT, "Burst averages p98 LOCAL")                     # step 7
THETA_ALL = os.path.join(DATA_OUT, "Theta signal all channels")                    # step 8
REP_SIG   = os.path.join(DATA_OUT, "Theta coupling representative channels")       # step 9
BETA_P90  = os.path.join(DATA_OUT, "Beta power p90 LOCAL")                         # step 10
PAIRS_DIR = os.path.join(AMIR, "Theta", "Phase connectivity", "All sessions all pairs")   # step 11 (if SAVE)

# ── switches ──
RUN       = False                                                                  # True = compute missing sessions (hours)
OVERWRITE = False                                                                  # True = recompute every session
SAVE      = False                                                                  # True = write figures and tables

# ── step parameters ──
TEST      = "20260529_r0000136_wt_1318"                                            # example session (step 6 keeps single bursts for it)
REF_SES   = "20260529_r0000136_wt_1318"                                            # step 5: session of the referencing figure
STEP_UM, MAX_HOURS = 90, 13                                                        # step 4: one channel per 90 µm per shank; stop after 13 h
PCTS      = (98,)                                                                  # step 6: burst threshold percentile(s); (98, 90) also makes p90

# ── code + data ──
sys.path.insert(0, CODE_DIR)
import fx_common as fxc, functions as fn, fx_analysis as fa
for m_ in (fxc, fn, fa): importlib.reload(m_)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
fxc.check_paths(PROC=PROC, TABLES=TABLES)
fxc.set_saving(SAVE, fig_dir=FIG_DIR, table_dir=TABLES)
final = pd.read_csv(os.path.join(TABLES, "final_sessions.csv"), dtype={"rat": str})
XB    = pd.read_csv(os.path.join(TABLES, "extra_bad_channels.csv"))
TR    = pd.read_csv(os.path.join(TABLES, "trials_filtered.csv"), dtype={"rat": str}, low_memory=False); K = TR[TR["keep"]].copy()
REP   = pd.read_csv(os.path.join(TABLES, "representative_channels.csv"))
CHREG = pd.read_csv(os.path.join(TABLES, "channel_regions_all.csv")); CHREG["channel_id"] = CHREG["channel_id"].astype(str)
print(f"final sessions: {len(final)} | kept trials: {len(K)} | extra bad channel rows: {len(XB)} (expected 107)")
print("representative channels:", REP[["rat", "region", "x", "y"]].to_dict("records"))

## Step 1 · Band power of all final sessions (CMR and LOCAL)

**What it does, per session and reference** (`functions.band_power_session`), with the kept trials (all `long_prob` blocks):
1. cuts epochs around the **press** (−2.5 to 3.5 s) and the **release** (−3.0 to 2.5 s), long enough for the slowest wavelets; only trials that fit in both are kept;
2. **Morlet wavelet power** in six bands, several frequencies per band: delta 1–3 Hz (2 cycles), theta 4–7 (5), alpha 8–12 (6), beta 13–29 (7),
   low gamma 30–60 (10, 50 Hz skipped), high gamma 61–100 Hz (10); 100 Hz time steps;
3. **per trial and channel, `logpow`:** log10 of the mean power (over the band's frequencies and the window) in fixed windows: before the press
   (−0.5 to −0.2 s), early hold (0.1–0.3 s), hold (0.2–0.85 s), before the release (−0.3–0 s), after the release (0.2–0.5 s), and before / after
   the tone (−0.3–0, 0–0.3 s; NaN without a tone);
4. **time courses, `curves`:** for each trial log10(power(t) / its own pre-press power, −0.5 to −0.2 s), kept only while the rat still holds;
   then the **mean over trials** per condition (correct / early / late × long / short, plus "early long held > 0.3 s"), press-aligned −0.7 to 2.0 s
   and release-aligned −1.5 to 0.6 s; `counts` = trials still holding at each time point;
5. **spectra:** Hann periodogram 1–100 Hz in the pre (−0.8 to −0.2 s), hold (0.25–0.85 s) and pre-tone (−0.6–0 s) windows, mean over trials.

**Why per trial and window:** any trial selection can then be tested in 07 without recomputing. About 3.8 min per session for CMR, 5.6 for LOCAL.

**Output:** one `.pkl` per session in `band power\CMR` and `band power\LOCAL` (keys: meta, trials, logpow, curves, counts, times, spectra, spec_n).

In [ ]:
fa.run_band_power(final, K, PROC, XB, BP_DIRS, run=RUN, overwrite=OVERWRITE, err_log=os.path.join(TABLES, "band_power_errors.csv"))

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

## Step 2 · Beta and gamma bursts (50-ms bins)

**What it does, per session** (`functions.bursts_session`), in the CMR and LOCAL reference:
1. **channels:** one representative channel per region (PL, IL, Cg1): the recorded channel closest to a fixed position per rat (rule 1.13b,
   `representative_channels.csv`); its distance to that position is saved;
2. **burst detection** (Karvat et al. 2021): Morlet power at every integer frequency of the band (beta 15–30 Hz, 7 cycles; gamma 30–100 Hz, 10 cycles,
   48–52 Hz set to 0 because of the notch); each frequency is divided by its own **90th percentile** over the session (artifact periods excluded);
   a burst = a region above 1 around a local maximum in time × frequency;
3. **artifacts:** samples above a fixed amplitude, padded, are excluded;
4. **per trial:** burst **occupancy** (fraction of each 50-ms bin covered by a burst) and **count** (burst peaks per bin), aligned to the press
   (−1.5 to 2.0 s), the tone (−1.0 to 0.5 s) and the release (−1.5 to 0.5 s).

Note: gamma is a much wider band, so its occupancy is higher than beta's; compare conditions **within** a band.

**Output:** `<session>.npz` + `<session>_trials.csv` in `Bursts detection`.

</div>

In [ ]:
fa.run_bursts(final, K, PROC, XB, REP, CHREG, BURST_DIR, run=RUN, overwrite=OVERWRITE, err_log=os.path.join(TABLES, "bursts_errors.csv"))

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

## Step 3 · Beta bursts with exact times

**What it does** (`functions.bursts_session_exact`): the same detection, channels and artifact rule as step 2, but it keeps the **exact start
and end time** of every beta burst and the time of every burst peak (session clock, s), plus the artifact periods, CMR and LOCAL.
Needed for "time from the last burst to the tone" (07, section 3). The old check showed that the occupancy rebuilt from these exact times
equals the 50-ms bins of step 2.

**Output:** `<session>.npz` + `<session>_trials.csv` in `Bursts detection exact times`.

</div>

In [ ]:
fa.run_bursts(final, K, PROC, XB, REP, CHREG, EXACT_DIR, run=RUN, overwrite=OVERWRITE, exact=True, err_log=os.path.join(TABLES, "bursts_exact_errors.csv"))

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

## Step 4 · Beta and gamma bursts over the whole probe (overnight)

**What it does:** the same burst rule as step 2 (LOCAL), but on a **grid of channels**: on every shank, the channel closest to the centre of each
`STEP_UM` (90 µm) depth bin, all labelled regions (PL, IL, Cg1, M2, basal forebrain). The 3 representative channels are always included as a check:
their result must equal the main burst files (the maximum difference is printed). Trials = the same rows as the main burst files.
Sessions alternate between rats, so a partial run covers all three; it stops starting new sessions after `MAX_HOURS` and can be restarted.

**Output:** `<session>.npz` in `Beta and gamma bursts across channels` (channel id, x, y, region; occupancy and counts as channels × trials × bins;
session occupancy; artifact fraction).

</div>

In [ ]:
fa.run_whole_probe_bursts(final, PROC, XB, CHREG, BURST_DIR, ALL_DIR, step_um=STEP_UM, max_hours=MAX_HOURS, run=RUN,
                          err_log=os.path.join(TABLES, "bursts_allch_errors.csv"))

## Step 5 · What the CMR and LOCAL references subtract (methods figure)

**What it does:** for one example channel in `REF_SES` (shank 2, 1875 µm): **A** the CMR subtracts the median of **all** recorded channels;
**B** LOCAL subtracts the median of the channels 30–200 µm around it (SpikeInterface's own neighbour list; if fewer than 5, the closest 5 beyond
30 µm). It also recomputes both references by hand for 1 s and prints the largest difference to SpikeInterface (should be below 1 bit).

**Saves:** `referencing example.png` in `PROBE_DIR` (it was a PDF before; now PNG).

In [ ]:
fig = fa.referencing_figure(PROC, XB, CHREG, REF_SES)
fxc.save_fig(fig, "referencing example", folder=PROBE_DIR); plt.show()

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

## Step 6 · Burst properties at the 98th percentile (Karvat et al. 2020)

**What it does, per session**, on the representative **IL** channel (LOCAL, beta 15–30 Hz, same wavelets and artifact rule as step 2), with the
**98th-percentile** threshold of Karvat et al. 2020 (`PCTS`):
- every burst **peak**: time, frequency, power × threshold, power × the median of that frequency (as Karvat 2020 Fig. 2e), frequency span
  (contiguous frequencies above threshold at the peak time) and duration (at the peak frequency);
- burst start / end times and artifact periods;
- **trough-aligned waveforms:** each peak is moved to the nearest trough of the 15–30 Hz filtered LFP (within half a period) and ±0.5 s of raw LFP is
  cut; saved as the mean of all bursts and of the 50 strongest (single examples for `TEST`);
- the task period (first press to last release), so 07 can keep bursts during the task only.

**Output:** `<session>.npz` in `Burst properties p98` (used by 07, section 3, steps 8–10).

</div>

In [ ]:
fa.run_burst_properties(final, K, PROC, XB, REP, CHREG, PROP_ROOT, pcts=PCTS, test=TEST, run=RUN, overwrite=OVERWRITE)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

## Step 7 · Average IL beta burst per session

**What it does, per session**, from the step-6 files (IL, LOCAL, 98th percentile):
1. **bursts** as in 07 step 8: the strongest peak of each merged burst, during the task only;
2. **selected bursts** (Karvat 2020): duration ≥ 70 ms and peak frequency 20–25 Hz; and **all** bursts;
3. each burst aligned to the beta trough nearest its peak, ±300 ms of raw LFP; bursts touching an artifact are left out;
4. the **mean** of the selected and of all bursts, plus 10 random selected single traces with their above-threshold mask;
5. per mean: **trough depth** (µV vs the baseline |t| > 200 ms), **oscillation length** (time the 15–30 Hz envelope stays above half its maximum;
   ms and cycles at 22.5 Hz), **clarity** (trough depth / baseline SD); and the **beta share** of the raw signal (15–30 / 1–200 Hz power).

**Output:** `<session>.npz` in `Burst averages p98 LOCAL` (used by 07, steps 9–10); the summary table is shown (and saved with `SAVE`).

</div>

In [ ]:
burst_avg = fa.run_burst_averages(PROC, XB, os.path.join(PROP_ROOT, "Burst properties p98"), AVG_DIR, run=RUN, overwrite=OVERWRITE)
fxc.save_table(burst_avg, "burst_average_metrics", folder=AVG_DIR)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## Step 8 · Theta signal of all channels (overnight)

**What it does, per session and reference** (CMR, LOCAL, original): every channel (after bad channels), every kept trial of type long or short with
label correct, early or late, from −1.5 to 2.0 s around the press in 20-ms steps. The **theta signal** = the sum of the complex Morlet coefficients at
4, 5, 6 and 7 Hz (5 cycles, as step 1), so each channel × trial × time has a theta amplitude and phase. Saved with channel ids, positions, regions and
the trial table, so 07 can compute coupling (PLV, PLI, PPC, Δφ) and theta power **for any channel pair** without recomputing. Trials are read in
batches and channels convolved in chunks (memory).

**Output:** `<ref>\<session>.npz` + `<session>_trials.csv` in `Theta signal all channels` (used by the all-pairs test, 07 section 2).

</div>

In [ ]:
fa.run_theta_all_channels(final, K, PROC, XB, CHREG, THETA_ALL, run=RUN, overwrite=OVERWRITE)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## Step 9 · Representative IL, PL, Cg1 signals in CMR and SR30-500

**What it does, per session:** the representative channel of each region (same as the burst files; distance to the fixed position saved, so 07 can
exclude channels far away), all kept trials, −2.5 to +3.5 s around the press at 1000 Hz, in two references built from the original recording:
- **CMR**: channel − median of all channels;
- **SR30-500**: channel − median of the channels of the **same region** 30–500 µm away. Why: no channel is shared between the IL and PL references
  and no partner-region channel enters, so shared signal can't create coupling between the regions; it was chosen in the pilot as the most local
  ring that still has a theta peak. The ring size is saved (fewer than 5 = flag).

**Output:** `<session>.npz` + `<session>_trials.csv` in `Theta coupling representative channels` (input of the coupling analyses, 07 section 2).

</div>

In [ ]:
fa.run_rep_channel_signals(final, K, PROC, XB, CHREG, BURST_DIR, REP_SIG, run=RUN, overwrite=OVERWRITE)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

## Step 10 · Beta power normalised to the 90th percentile (Karvat et al. 2021)

**What it does, per session**, on the representative channels (LOCAL): power at every frequency 15–30 Hz (7 cycles, as the bursts), each divided by
its own artifact-free **90th percentile** (= the burst threshold, so 1 means "at threshold"), averaged over frequencies; then cut −1.0 to 0.5 s around
the tone of every trial (10-ms steps), same trial rows as the burst files.

**Output:** `<session>.npz` in `Beta power p90 LOCAL` (used by 07, section 3, beta power around the tone).

</div>

In [ ]:
fa.run_beta_power_p90(PROC, XB, BURST_DIR, BETA_P90, run=RUN, overwrite=OVERWRITE)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## Step 11 · Channels for the all-pairs IL–PL test

**What it does, per session:** chooses the IL and PL channels used in the all-pairs coupling test (07, section 2):
1. **usable** = at least 5 channels of the same region in its 30–500 µm ring (so the SR30-500 reference is defined);
2. every **4th** usable channel per region (sorted by shank, depth, x), to keep the number of pairs manageable;
3. only channels that have a partner of the other region at least **1000 µm** away (so neighbouring channels can't share signal).

Prints the table (usable / tested channels and pairs per session, per rat) and draws one panel per session.

**Saves:** `all_pairs_channels_per_session.csv` and `.png` in `PAIRS_DIR`.

</div>

In [ ]:
pairs_sel, pairs_geo = fa.all_pairs_selection(CHREG, XB, final)
fig = fa.plot_all_pairs_selection(pairs_sel, pairs_geo)
fxc.save_table(pairs_sel, "all_pairs_channels_per_session", folder=PAIRS_DIR); fxc.save_fig(fig, "all_pairs_channels_per_session", folder=PAIRS_DIR); plt.show()

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## Steps 12–16 · Inputs of the coupling tests and the multi-unit firing (recovered code)

These steps use the same engine as 07 (`fx_results.py`) and the same files (`fx_res_coupling.py`, `fx_res_power.py`).
The next cell connects them to the settings of this notebook (no computing). Each step computes only when `RUN = True` and skips finished sessions.
Shared settings of the coupling (in `fx_res_coupling.py`): the three IL–PL pairs (one per rat, `PAIRS3`), their sessions (`BACKUP_A_C11_C10_rows.csv`),
same-region ring reference 30–500 µm (SR30-500, ≥ 5 ring channels), theta 4–7 Hz Morlet (5 cycles), 10-ms steps, early trials held ≥ 0.3 s, ≥ 5 trials.
A log of every session is written to `Tests\ALL_overnight_log.txt`.

</div>

In [ ]:
import fx_results as fxr, fx_res_coupling as fxco, fx_res_power as fxp
fxr.setup(CODE_DIR, DATA_ROOT, TABLES, FIG_DIR, TABLES, SAVE=SAVE, RUN=RUN)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## Step 12 · Per-trial IL / PL theta phases around the press

**What it does, per session** of the three chosen pairs: correct, early (held ≥ 0.3 s) and late long trials; IL and PL with the SR30-500 reference;
Morlet 4–7 Hz (5 cycles) from −0.5 to 1.1 s around the press in 10-ms steps; saves the **unit phase** (complex, length 1) of every trial, time,
trial index, outcome label, hold time and rat. ERP kept.

**Why:** many coupling cells of 07 (bias-corrected slide figure, coupling over the trial, onset consistency, late trials, coupling along the session)
re-use the same phases; computing them once saves hours.

**Output:** `<session>.npz` in `Tests\Trial_phases_press_SR30-500_ERPkept`.

</div>

In [ ]:
fxr.run(fxco.trial_phase_files)

<div style="color:#FFA000; border-left:4px solid #FFA000; padding-left:10px">

## Step 13 · Jäckel et al. 2024 Fig. 3C: sums of the phase-difference vectors, ERP removed

**What it does, per session** with ≥ 5 correct and ≥ 5 early long trials: IL and PL (SR30-500) around the press and the release (−1 to +1 s, 10-ms steps);
the ERP (mean over the trials of the condition) is subtracted per condition before filtering; FIR band-pass (delta 2–4 Hz, theta 4–8 Hz; filter lengths as the paper)
+ Hilbert phase; per time point the **sum** of the unit phase-difference vectors over the trials and the number of trials.

**Why:** with the sums, sessions can be pooled per rat exactly and the unbiased PPC computed (07: T8c).

**Output:** `<session>.npz` in `Tests\T8c_Jaeckel_Fig3C_replication_SR30-500_ERPremoved`.

</div>

In [ ]:
fxr.run(fxco.T8c_sums)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## Step 14 · Overnight coupling tests: T8 θ and δ, T11, T13–15

**T8** (theta 4–8 Hz, delta 2–4 Hz) — Jäckel replication per session: FIR + Hilbert, PLV / PLI around the press and the release, correct vs early,
draws of equal trial numbers and trial shuffles; trace figure `F1_traces.png` and `T8_summary.txt`.
**T11** — is the coupling explained by theta amplitude? Per session, coupling in high- vs low-amplitude trials; `T11_sessions_SR30-500.csv`, `T11_summary.txt`.
**T13–15** — coupling in windows with equal trial numbers (ERP kept): T13 before the tone, T14 0.2-s windows after the press (used by the timeline cells
of 07), T15 around the release; `T13_rows.csv`, `T14_rows.csv`, `T15_rows.csv`, `T13-15_summary.txt`.
Tests in the summaries: sign-flip with every rat weighted equally (one-sided vs chance; two-sided for differences).

Each part runs alone: an error is logged and the next part runs. Resumes (finished sessions are skipped). Hours → overnight.

</div>

In [ ]:
fxr.run(fxco.overnight_tests)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## Step 15 · T13 fix: theta coupling before the tone (−0.5 → 0 s), correct vs late, ERP kept

The first T13 had no rows (the tone column was read wrongly); this step checks the tone column and computes T13 again:
correct vs late long trials, −0.5 to 0 s before the tone, equal trial numbers (30 draws), 100 shuffles. Output: `T13_rows_fix.csv`, `T13_summary_fix.txt`.

</div>

In [ ]:
fxr.run(fxco.T13_fix)

<div style="color:#6a1b9a; border-left:4px solid #6a1b9a; padding-left:10px">

## Step 16 · Multi-unit firing over time (spike-rate curves)

**What it does, per session:** raw 30 kHz AP band (band-pass 300–6000 Hz, phase shift, CMR); spikes = downward crossings of −4.5 × noise (MAD / 0.6745),
1-ms dead time; correct long trials; firing rate in 50-ms bins around the press and the release, per mPFC channel with ≥ 1 spike/s before the press;
log10(rate / own pre-press rate). Also draws the high gamma + spikes figure once all sessions are done.

**Output:** one file per session in `tables\mua_curves` (07, section 1: high gamma + spikes over time). Slow (raw data).

</div>

In [ ]:
fxr.run(fxp.mua_curves)